# NASA C-MAPSS Turbofan Engine RUL Prediction
## Notebook 02: Preprocessing, RUL Capping, and Feature Engineering

### Objective
In this notebook, we transform raw multi-sensor telemetry into clean feature matrices and sequence tensors:
1. **RUL Target Calculation**: Compute run-to-failure cycles for each engine.
2. **Piecewise-Linear RUL Capping**: Cap RUL at 125 cycles to align loss with the physical degradation knee.
3. **Grouped Engine Split**: Split 80/20 by engine ID to ensure zero data leakage between train and validation.
4. **Leak-Free MinMax Normalization**: Fit `MinMaxScaler` strictly on the training fold, then scale validation and test data.
5. **Feature Engineering for Classical ML**: Generate rolling mean and rolling std features over a 30-cycle window.
6. **Sliding Windows for Deep Learning**: Construct 3D sequence tensors `(N, 30, 14)` for PyTorch recurrent networks.

In [ ]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_loader import load_dataset
from src.preprocess import add_rul_targets, split_train_val_by_engine, fit_and_apply_scaler
from src.features import create_tabular_features, create_sequence_windows
from config.config import FD001_ACTIVE_SENSORS, WINDOW_SIZE, RUL_CAP

print("Preprocessing modules loaded.")

### 1. Compute Ground-Truth RUL & Apply Piecewise Capping
In early cycles, engines operate without detectable damage. Predicting an uncapped RUL of 280 vs 230 forces models to fit random sensor jitter. Capping at 125 focuses learning on the actual degradation regime.

In [ ]:
train_raw, test_raw, rul_true = load_dataset("FD001")
train_with_rul = add_rul_targets(train_raw, rul_cap=RUL_CAP)

unit_1 = train_with_rul[train_with_rul["unit_number"] == 1]
plt.figure(figsize=(8, 3.5))
plt.plot(unit_1["time_cycles"], unit_1["RUL_true"], label="Uncapped Linear RUL", color="gray", linestyle="--")
plt.plot(unit_1["time_cycles"], unit_1["RUL_capped"], label="Piecewise Capped RUL (125)", color="#1b9e77", linewidth=2)
plt.title("Engine 1: Linear vs Piecewise-Linear RUL Target")
plt.xlabel("Operating Cycles")
plt.ylabel("Remaining Useful Life (Cycles)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.6)
plt.show()

### 2. Grouped Engine Split & Leak-Free Scaling
We split 80 engines for training and 20 engines for validation. `MinMaxScaler` is fitted only on the 80 training engines.

In [ ]:
train_split, val_split, train_ids, val_ids = split_train_val_by_engine(train_with_rul)
print(f"Train Engines ({len(train_ids)}): {train_ids[:10]}...")
print(f"Validation Engines ({len(val_ids)}): {val_ids}")

train_scaled, val_scaled, test_scaled, scaler = fit_and_apply_scaler(
    train_split, val_split, test_raw, feature_cols=FD001_ACTIVE_SENSORS
)
print("MinMax Scaler fitted successfully on training engines.")

### 3. Feature Engineering: Rolling Statistics for Classical Models
We calculate rolling mean and rolling standard deviation over a 30-cycle window for each of the 14 active sensors.

In [ ]:
train_eng = create_tabular_features(train_scaled, FD001_ACTIVE_SENSORS, WINDOW_SIZE)
print(f"Engineered shape: {train_eng.shape}")
sample_cols = ["time_cycles", "s_2", f"s_2_roll_mean_{WINDOW_SIZE}", f"s_2_roll_std_{WINDOW_SIZE}"]
train_eng[train_eng["unit_number"] == 1][sample_cols].iloc[25:35]

### 4. Sequence Windowing for Deep Learning
We create 3D tensors of shape `(N, 30, 14)` for sequence models.

In [ ]:
X_seq, y_seq = create_sequence_windows(train_scaled, FD001_ACTIVE_SENSORS, WINDOW_SIZE)
print(f"Sequence Tensor Shape: {X_seq.shape}")
print(f"Sequence Target Shape: {y_seq.shape}")
print(f"First window cycle range: 1 to 30, target RUL: {y_seq[0]}")